In [ ]:

import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
torch.set_num_threads(2)

DEVICE = "cpu"
DIM = 4
N_PATCHES = 4
BATCH = 16

# ---------------------------------------------------
# 1. Structured synthetic images
# --------------------------------------------------

def make_images(n):
    images = torch.zeros(n, 1, 8, 8)

    for i in range(n):
        # Smooth horizontal and vertical structure,
        # with varying amplitude and phase.
        a = torch.rand(1).item() * 0.6 + 0.2
        b = torch.rand(1).item() * 0.6 + 0.2
        x = torch.arange(8).float() / 7
        y = torch.arange(8).float() / 7

        pattern = (
            a * x[None, :] +
            b * y[:, None]
        )
        images[i, 0] = pattern

    return images

images = make_images(128)

# --------------------------------------------------
# 2. Tiny patch-based Transformer encoder
# --------------------------------------------------

class TinyEncoder(nn.Module):
    def __init__(self, dim=4):
        super().__init__()

        # Four 4x4 patches from each 8x8 image
        self.patch_embed = nn.Linear(16, dim)
        self.pos_embed = nn.Parameter(
            torch.randn(1, 4, dim) * 0.02
        )

        layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=2,
            dim_feedforward=16,
            dropout=0.0,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer, num_layers=1,
            enable_nested_tensor=False
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, patches, indices):
        # patches: (batch, 4, 16)
        selected = patches[:, indices, :]
        tokens = (
            self.patch_embed(selected) +
            self.pos_embed[:, indices, :]
        )
        return self.norm(self.transformer(tokens))

# --------------------------------------------------
# 3. Classical latent predictor
# --------------------------------------------------

class ClassicalPredictor(nn.Module):
    def __init__(self, dim=4):
        super().__init__()

        self.target_pos = nn.Embedding(4, dim)

        self.network = nn.Sequential(
            nn.Linear(dim * 2, 16),
            nn.GELU(),
            nn.Linear(16, dim)
        )

    def forward(self, context, target_idx):
        # Mean of visible context tokens
        context_summary = context.mean(dim=1)

        target_positions = torch.full(
            (context.shape[0],),
            target_idx,
            dtype=torch.long,
            device=context.device
        )
        pos = self.target_pos(target_positions)

        return self.network(
            torch.cat([context_summary, pos], dim=-1)
        )

# --------------------------------------------------
# 4. Build JEPA components
# --------------------------------------------------

encoder = TinyEncoder(DIM).to(DEVICE)
target_encoder = copy.deepcopy(encoder)
predictor = ClassicalPredictor(DIM).to(DEVICE)

for p in target_encoder.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.Adam(
    list(encoder.parameters()) +
    list(predictor.parameters()),
    lr=0.002
)

# --------------------------------------------------
# 5. Image to patches
# --------------------------------------------------

def patchify(images):
    # (B, 1, 8, 8) -> (B, 4, 16)
    patches = F.unfold(
        images, kernel_size=4, stride=4
    )
    return patches.transpose(1, 2)

# --------------------------------------------------
# 6. Full JEPA training loop
# --------------------------------------------------

EMA_MOMENTUM = 0.99

print("Starting classical JEPA training")

for epoch in range(30):
    permutation = torch.randperm(len(images))
    total_loss = 0.0

    for start in range(0, len(images), BATCH):
        batch_indices = permutation[start:start+BATCH]
        batch = images[batch_indices].to(DEVICE)

        patches = patchify(batch)

        # Select one hidden target patch
        target_idx = torch.randint(
            0, N_PATCHES, (1,)
        ).item()

        context_indices = [
            i for i in range(N_PATCHES)
            if i != target_idx
        ]

        # Context encoder sees only visible patches
        context = encoder(
            patches, context_indices
        )

        # Target encoder sees the full image
        with torch.no_grad():
            all_targets = target_encoder(
                patches, list(range(N_PATCHES))
            )
            target = all_targets[:, target_idx, :]

        # Predict hidden latent representation
        prediction = predictor(
            context, target_idx
        )

        loss = F.smooth_l1_loss(
            prediction, target
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # EMA target encoder update
        with torch.no_grad():
            for p, tp in zip(
                encoder.parameters(),
                target_encoder.parameters()
            ):
                tp.mul_(EMA_MOMENTUM).add_(
                    p, alpha=1 - EMA_MOMENTUM
                )

        total_loss += loss.item() * len(batch)

    mean_loss = total_loss / len(images)

    if epoch % 5 == 0 or epoch == 29:
        print(
            f"Epoch {epoch:02d} | "
            f"Loss: {mean_loss:.6f}"
        )

print("\nClassical JEPA execution completed.")

# --------------------------------------------------
# 7. Final verification
# --------------------------------------------------

patches = patchify(images[:4])
context = encoder(patches, [0, 1, 2])
prediction = predictor(context, 3)

with torch.no_grad():
    targets = target_encoder(
        patches, [0, 1, 2, 3]
    )[:, 3, :]

print("Context shape:", tuple(context.shape))
print("Prediction shape:", tuple(prediction.shape))
print("Target shape:", tuple(targets.shape))

assert context.shape == (4, 3, 4)
assert prediction.shape == (4, 4)
assert targets.shape == (4, 4)

print("Architecture shape checks: PASSED")


Starting classical JEPA training
Epoch 00 | Loss: 0.442964
Epoch 05 | Loss: 0.204717
Epoch 10 | Loss: 0.080325
Epoch 15 | Loss: 0.037688
Epoch 20 | Loss: 0.014485
Epoch 25 | Loss: 0.013626
Epoch 29 | Loss: 0.012474

Classical JEPA execution completed.
Context shape: (4, 3, 4)
Prediction shape: (4, 4)
Target shape: (4, 4)
Architecture shape checks: PASSED
